<div dir="rtl">

# الترم ٣ · الوحدة ٢: الرؤية بالكمبيوتر (Computer Vision)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-3/09-computer-vision.ipynb)

**الصورة = مصفوفة أرقام.** هنفتح صورة، ونلعب بالبكسلات، ونعمل فلاتر بإيدينا، ونفهم **الـ convolution** اللي هي قلب كل موديلات الصور الحديثة (CNNs).

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

<div dir="rtl">

## ١. الصورة من جوه

</div>

In [ ]:
img = load_sample_image("china.jpg")
print("Shape (height, width, channels):", img.shape, "| dtype:", img.dtype)
print("Top-left pixel (R, G, B):", img[0, 0])

plt.figure(figsize=(7, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

<div dir="rtl">

كل بكسل فيه ٣ أرقام من ٠ لـ ٢٥٥: **أحمر، أخضر، أزرق (RGB)**.

</div>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, (ax, name, cmap) in enumerate(zip(axes, ["Red", "Green", "Blue"], ["Reds", "Greens", "Blues"])):
    ax.imshow(img[:, :, i], cmap=cmap)
    ax.set_title(name)
    ax.axis("off")
plt.show()

<div dir="rtl">

## ٢. فلاتر بسيطة = عمليات حسابية

</div>

In [ ]:
gray = img.mean(axis=2)                                  # رمادي: متوسط الألوان
bright = np.clip(img.astype(int) + 60, 0, 255).astype(np.uint8)
negative = 255 - img
crop = img[100:300, 200:450]
flipped = img[:, ::-1]

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, (title, im) in zip(axes, [("Gray", gray), ("Brighter", bright), ("Negative", negative),
                                  ("Crop", crop), ("Flip", flipped)]):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
    ax.set_title(title)
    ax.axis("off")
plt.show()

<div dir="rtl">

## ٣. الـ Convolution: الفلتر اللي بيمشي على الصورة

بناخد مربع صغير من الأرقام (**kernel**)، ونحركه على الصورة. في كل مكان نضرب ونجمع. الـ kernel بيحدد الفلتر هيعمل إيه.

</div>

In [ ]:
def convolve(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """2D convolution (valid mode) written with NumPy, no libraries."""
    kh, kw = kernel.shape
    h, w = image.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for dy in range(kh):
        for dx in range(kw):
            out += kernel[dy, dx] * image[dy:dy + h - kh + 1, dx:dx + w - kw + 1]
    return out

KERNELS = {
    "Blur": np.ones((5, 5)) / 25,
    "Sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "Vertical edges": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Horizontal edges": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, k) in zip(axes, KERNELS.items()):
    result = convolve(gray, k)
    if "edges" in name:
        ax.imshow(np.abs(result), cmap="gray")
    else:
        ax.imshow(np.clip(result, 0, 255), cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.show()

<div dir="rtl">

**الفكرة الكبيرة:** في الـ CNN، الشبكة **بتتعلم الـ kernels لوحدها**. أول طبقات بتتعلم حواف، وبعدها أشكال، وبعدها عيون وعجل وودان... لحد ما تعرف "دي قطة".

</div>

<div dir="rtl">

## ٤. كاشف الحواف الكامل (Sobel)

</div>

In [ ]:
gx = convolve(gray, KERNELS["Vertical edges"])
gy = convolve(gray, KERNELS["Horizontal edges"])
edges = np.hypot(gx, gy)
plt.figure(figsize=(8, 5))
plt.imshow(edges > np.percentile(edges, 85), cmap="gray_r")
plt.title("Edges (Sobel)")
plt.axis("off")
plt.show()

<div dir="rtl">

## ٥. نلاقي حاجة في الصورة بالألوان

أبسط "object detection": نعلّم البكسلات اللي لونها قريب من لون معين (هنا: أزرق السما)، ونرسم مربع حواليها.

**مشكلة:** بكسلات قليلة متفرقة (انعكاس المية مثلاً) ممكن تكبّر المربع على الصورة كلها. علشان كده بنتجاهل أطراف التوزيع (أبعد ٢٪ من كل ناحية).

</div>

In [ ]:
def find_color(image, target_rgb, tolerance=40, trim=2):
    dist = np.linalg.norm(image.astype(int) - np.array(target_rgb), axis=2)
    mask = dist < tolerance
    if not mask.any():
        return mask, None
    ys, xs = np.where(mask)
    x0, x1 = np.percentile(xs, [trim, 100 - trim])
    y0, y1 = np.percentile(ys, [trim, 100 - trim])
    return mask, (x0, y0, x1, y1)

sky_mask, box = find_color(img, target_rgb=(170, 200, 230), tolerance=35)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].imshow(sky_mask, cmap="Blues")
axes[0].set_title("Mask: pixels close to sky-blue")
axes[1].imshow(img)
if box:
    x0, y0, x1, y1 = box
    axes[1].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="red", linewidth=2))
axes[1].set_title("Detected region")
for ax in axes:
    ax.axis("off")
plt.show()
print(f"{sky_mask.mean():.0%} of the image is sky-blue")

<div dir="rtl">

الطريقة دي بسيطة وبتتلخبط بسهولة (المية لونها قريب من السما!). الموديلات الحقيقية زي YOLO **بتتعلم** شكل الحاجة مش لونها بس.

</div>

<div dir="rtl">

## 🏋️ تمرين ١: فلتر "Emboss"
جرّب الـ kernel ده: `[[-2, -1, 0], [-1, 1, 1], [0, 1, 2]]` واعرض النتيجة.

<details><summary>الحل</summary>

```python
emboss = np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]])
plt.imshow(convolve(gray, emboss), cmap="gray")
plt.axis("off")
plt.show()
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🏋️ تمرين ٢: Pixelate
اعمل الصورة "مربعات" (زي الألعاب القديمة): خد كل بكسل عاشر وكبّره.

**تلميح:** `img[::10, ::10]` و `np.repeat`.

<details><summary>الحل</summary>

```python
small = img[::10, ::10]
pixel = np.repeat(np.repeat(small, 10, axis=0), 10, axis=1)
plt.imshow(pixel)
plt.axis("off")
plt.show()
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🔭 الخطوة الجاية
- **OpenCV** (`cv2`): مكتبة فيها كل الفلاتر دي وأسرع بكتير، وبتشتغل على فيديو الكاميرا.
- **YOLO**: موديل بيلاقي عشرات الحاجات في الصورة في نفس اللحظة. جرّبه في Colab:

```python
!pip install -q ultralytics
from ultralytics import YOLO
YOLO("yolo11n.pt")("https://ultralytics.com/images/bus.jpg")[0].show()
```

## ✅ لخّص
- الصورة = مصفوفة (height × width × 3).
- الفلاتر = عمليات حسابية على البكسلات.
- **Convolution** = kernel صغير بيمشي على الصورة. الـ CNN بتتعلم الـ kernels.

</div>